In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.font_manager import FontProperties, fontManager

# Figure 4: Distribution of derived agroclimatic indicators

# Repository root
ROOT_DIR = Path.cwd()

if ROOT_DIR.name == "figure_scripts":
    ROOT_DIR = ROOT_DIR.parent

# Input data
DATA_FILE = (
    ROOT_DIR
    / "dataset"
    / "data"
    / "processed"
    / "agroclimate_with_indicators.parquet"
)

# Output directory
OUTPUT_DIR = ROOT_DIR / "figure_scripts" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PNG_FILE = OUTPUT_DIR / "figure4_derived_indicator_distributions.png"
PDF_FILE = OUTPUT_DIR / "figure4_derived_indicator_distributions.pdf"

# Configure font
FONT_FILE = ROOT_DIR / "calibri.ttf"

if FONT_FILE.exists():
    fontManager.addfont(str(FONT_FILE))
    calibri = FontProperties(fname=str(FONT_FILE))
    plt.rcParams["font.family"] = calibri.get_name()
else:
    calibri = FontProperties(family="DejaVu Sans")
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["font.size"] = 16

# Load derived indicators
columns = [
    "rainfall_7d",
    "rainfall_30d",
    "rainfall_90d",
    "rainy_days_30d",
    "rainy_days_90d",
    "max_daily_rainfall_30d",
    "consecutive_dry_days",
    "temperature_range",
    "rolling_temperature_mean_7d",
    "cumulative_et0_30d",
    "water_balance",
    "rolling_water_balance_30d"
]

data = pd.read_parquet(DATA_FILE, columns=columns)

# Random sample for faster histogram plotting
data = data.sample(
    n=min(200_000, len(data)),
    random_state=42
)

# Configure variables, axis labels, and panel titles
variables = [
    ("rainfall_7d", "Rainfall (mm)", "7-Day Rainfall Accumulation"),
    ("rainfall_30d", "Rainfall (mm)", "30-Day Rainfall Accumulation"),
    ("rainfall_90d", "Rainfall (mm)", "90-Day Rainfall Accumulation"),
    ("rainy_days_30d", "Number of days", "30-Day Rainy-Day Count"),
    ("rainy_days_90d", "Number of days", "90-Day Rainy-Day Count"),
    ("max_daily_rainfall_30d", "Rainfall (mm)", "Maximum Daily Rainfall"),
    ("consecutive_dry_days", "Number of days", "Consecutive Dry-Spell Length"),
    ("temperature_range", "Temperature range (°C)", "Diurnal Temperature Range"),
    ("rolling_temperature_mean_7d", "Temperature (°C)", "7-Day Mean Temperature"),
    ("cumulative_et0_30d", "ET₀ (mm)", "30-Day Cumulative ET₀"),
    ("water_balance", "Water balance (mm)", "Daily Water Balance"),
    ("rolling_water_balance_30d", "Water balance (mm)", "30-Day Water Balance")
]

# Create histogram plots
fig, axes = plt.subplots(6, 2, figsize=(12, 20))
axes = axes.flatten()

for ax, (column, xlabel, panel_title) in zip(axes, variables):
    values = data[column].dropna()

    ax.hist(
        values,
        bins=50,
        edgecolor="black",
        linewidth=0.4
    )

    ax.set_title(
        panel_title,
        loc="left",
        fontproperties=calibri,
        fontsize=14,
        pad=8
    )

    ax.set_xlabel(
        xlabel,
        fontproperties=calibri,
        fontsize=16
    )
    ax.set_ylabel(
        "Frequency",
        fontproperties=calibri,
        fontsize=16
    )
    ax.tick_params(axis="both", labelsize=14)

    for tick_label in ax.get_xticklabels() + ax.get_yticklabels():
        tick_label.set_fontproperties(calibri)
        tick_label.set_fontsize(14)

    ax.grid(
        axis="y",
        linestyle="--",
        linewidth=0.4,
        alpha=0.4
    )

# Adjust layout and save figures
plt.tight_layout(h_pad=1.5, w_pad=1.0)

fig.savefig(PNG_FILE, dpi=300, bbox_inches="tight")
fig.savefig(PDF_FILE, format="pdf", bbox_inches="tight")

plt.show()
plt.close(fig)

print("\nFigure 4 saved.")
print("PNG:", PNG_FILE)
print("PDF:", PDF_FILE)